<a href="https://colab.research.google.com/github/bauer-san/CCRT-route-planner/blob/main/Route_Planner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ortools pandas googlemaps

  Preparing metadata (setup.py) ... done
  Created wheel for googlemaps: filename=googlemaps-4.10.0-py3-none-any.whl size=40714 sha256=aa3c8714673ebb119607ead24cbac783852f9bec6f10928de03493b0511e6d5e
  Stored in directory: /root/.cache/pip/wheels/4c/6a/a7/bbc6f5c200032025ee655deb5e163ce8594fa05e67d973aad6
Successfully built googlemaps


In [4]:
## THIS OPTIMIZATION WORKS AS INTENDED
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import googlemaps
from google.colab import userdata
from pprint import pprint
import urllib.parse

# Initialize Google Maps client (assuming API key is loaded from secrets in a previous cell)
gmaps = googlemaps.Client(key=userdata.get('GOOGLE_MAPS_API_KEY'))

# 1. DATA PREPARATION
def create_data_model(addresses, num_vehicles, depot_index=0):
    data = {}
    data['addresses'] = addresses
    data['num_vehicles'] = num_vehicles
    data['depot'] = depot_index

    num_addresses = len(addresses)
    full_distance_matrix = [[0 for _ in range(num_addresses)] for _ in range(num_addresses)]
    chunk_size = 10 # To respect the 100 elements (origin * destination) limit per API call

    print("Fetching distance matrix from Google Maps API in chunks...")

    try:
        for i in range(0, num_addresses, chunk_size):
            origin_chunk = addresses[i:i + chunk_size]
            for j in range(0, num_addresses, chunk_size):
                destination_chunk = addresses[j:j + chunk_size]

                matrix_result = gmaps.distance_matrix(
                    origins=origin_chunk,
                    destinations=destination_chunk,
                    mode="driving",
                    units="metric"
                )

                if matrix_result['status'] == 'OK':
                    for idx_origin, origin in enumerate(matrix_result['rows']):
                        for idx_dest, element in enumerate(origin['elements']):
                            global_origin_index = i + idx_origin
                            global_destination_index = j + idx_dest

                            if element['status'] == 'OK':
                                # Distance is in meters
                                full_distance_matrix[global_origin_index][global_destination_index] = element['distance']['value']
                            else:
                                # Placeholder for unreachable or error
                                full_distance_matrix[global_origin_index][global_destination_index] = 999999999
                else:
                    print(f"Error fetching distance matrix for chunk (origins {i}-{i+len(origin_chunk)-1}, destinations {j}-{j+len(destination_chunk)-1}): {matrix_result['status']}")
                    # If a chunk fails, we might want to fill with a large value or raise an error
                    # For simplicity, filling the failed chunk area with 999999999 for all elements
                    for idx_origin in range(len(origin_chunk)):
                        for idx_dest in range(len(destination_chunk)):
                            global_origin_index = i + idx_origin
                            global_destination_index = j + idx_dest
                            full_distance_matrix[global_origin_index][global_destination_index] = 999999999

        data['distance_matrix'] = full_distance_matrix
        print("Distance matrix fetched and compiled successfully.")

    except Exception as e:
        print(f"An unexpected error occurred during API calls: {e}")
        # Fallback to a dummy matrix or raise an error
        data['distance_matrix'] = [[0 for _ in addresses] for _ in addresses] # Dummy matrix
    pprint(data['distance_matrix'])
    return data

# 2. THE SOLVER (The "Brain")
def solve_routing(data):
    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'], data['depot'])
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Balancing load (max distance per team). Adjust this value based on typical route lengths.
    # The distance is in meters (converted to km in create_data_model, so 5000 means 5000km)
    routing.AddDimension(transit_callback_index, 0, 5000000, True, 'Distance') # Changed max capacity to 5000km in meters

    # Add objective: Minimize the maximum distance traveled by any vehicle.
    distance_dimension = routing.GetDimensionOrDie('Distance')
    # Add a global span constraint to the distance dimension (optional, but good practice)
    # distance_dimension.SetGlobalSpanCostCoefficient(100) # This is for minimizing span, not max

    # Minimize the maximum distance traveled by any vehicle.
    # We need to find the maximum cumul value among all vehicles at their respective end nodes.
    # The objective is to minimize this maximum value.
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)
    # Add a dimension for distance
    distance_dimension = routing.GetDimensionOrDie('Distance')
    distance_dimension.SetGlobalSpanCostCoefficient(100)

    search_params = pywrapcp.DefaultRoutingSearchParameters()
    search_params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    return routing, manager, routing.SolveWithParameters(search_params)

# 3. THE FORMATTER (The "Output")
def get_readable_output(data, manager, routing, solution):
    results = {}
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route_for_team = []
        route_distance = 0
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_for_team.append(data['addresses'][node_index])
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)

        # Add the final return to depot
        route_for_team.append(data['addresses'][manager.IndexToNode(index)])
        results[f"Team {vehicle_id + 1}"] = {
            'route': route_for_team,
            'distance': route_distance # Distance in the unit returned by callback (km in this case)
        }
    return results

def print_final_manifests(route_results):
    for team, details in route_results.items():
        stops = details['route']
        distance = details['distance']
        print(f"\n=== {team.upper()} MANIFEST ===")
        print(f"Total Distance: {distance} meters") # Updated to meters
        # Generate Google Maps Link
        encoded_waypoints = [urllib.parse.quote_plus(stop) for stop in stops[1:-1]] # URL encode waypoints
        waypoints_str = "|".join(encoded_waypoints)
        gmaps_url = f"https://www.google.com/maps/dir/?api=1&origin={urllib.parse.quote_plus(stops[0])}&destination={urllib.parse.quote_plus(stops[-1])}&waypoints={waypoints_str}"

        print(f"Digital Route: {gmaps_url}")
        print(f"{'Stop #':<8} | {'Address'}")
        print("-" * 40)
        for i, addr in enumerate(stops):
            label = "START" if i == 0 else "END" if i == len(stops)-1 else f"Stop {i}"
            print(f"{label:<8} | {addr}")

# 4. EXECUTION
if __name__ == '__main__':
    # Example usage with your addresses
    original_sample_addresses = [
        "49 Clinton St, Pontiac, MI, USA",
        "53 Waldo St, Pontiac, MI, USA",
        "3000 Winston Dr, Apt 3108, Pontiac, MI, USA",
        "60 Mechanic St, Pontiac, MI, USA"
    ]
    sample_addresses = [
    "6 N Saginaw St, Pontiac, MI, USA",
    "22 N Saginaw St, Pontiac, MI, USA",
    "51 N Saginaw St, Pontiac, MI, USA",
    "67 N Saginaw St, Pontiac, MI, USA",
    "108 N Saginaw St, Pontiac, MI, USA",
    "180 N Saginaw St, Pontiac, MI, USA",
    "20 Franklin Rd, Pontiac, MI, USA",
    "18 W Huron St, Pontiac, MI, USA",
    "47450 Woodward Ave, Pontiac, MI, USA",
    "44405 Woodward Ave, Pontiac, MI, USA",
    "60 E Pike St, Pontiac, MI, USA",
    "51000 Woodward Ave, Pontiac, MI, USA",
    "1275 N Perry St, Pontiac, MI, USA",
    "1051 Arlene Ave, Pontiac, MI, USA",
    "154 Lake St, Pontiac, MI, USA",
    "312 S Edith St, Pontiac, MI, USA",
    "858 Amanda Ln, Pontiac, MI, USA",
    "256 Cedardale Ave, Pontiac, MI, USA",
    "520 Linda Vista Dr, Pontiac, MI, USA",
    "160 Chippewa Rd, Pontiac, MI, USA",
    "215 Oak Ridge Dr, Pontiac, MI, USA",
    "738 Arusha Dr, Pontiac, MI, USA",
    "20 Park Pl, Pontiac, MI, USA",
    "180 W Rutgers Ave, Pontiac, MI, USA",
    "635 E Tennyson Ave, Pontiac, MI, USA"
]
    num_teams = 3

    main_data = create_data_model(sample_addresses, num_teams)

    if 'distance_matrix' in main_data and main_data['distance_matrix']:
        routing_model, routing_manager, solution_obj = solve_routing(main_data)

        if solution_obj:
            readable_routes = get_readable_output(main_data, routing_manager, routing_model, solution_obj)
            print_final_manifests(readable_routes)
        else:
            print("No solution found!")
    else:
        print("Could not create data model due to missing distance matrix. Check API key and network connection.")

Fetching distance matrix from Google Maps API in chunks...
Distance matrix fetched and compiled successfully.
[[0,
  154,
  163,
  240,
  449,
  536,
  1951,
  289,
  1081,
  2976,
  430,
  685,
  4133,
  4156,
  2283,
  3295,
  6176,
  3355,
  2827,
  2838,
  2788,
  3155,
  1551,
  4211,
  3855],
 [154,
  0,
  317,
  357,
  448,
  653,
  2054,
  406,
  1236,
  3079,
  397,
  788,
  4021,
  4044,
  2385,
  3203,
  6064,
  3458,
  2713,
  2955,
  2891,
  3271,
  1654,
  4287,
  3743],
 [163,
  317,
  0,
  77,
  286,
  373,
  2114,
  126,
  1187,
  3139,
  593,
  848,
  3970,
  3993,
  2445,
  3154,
  6013,
  3518,
  2664,
  2675,
  2951,
  2992,
  1714,
  4048,
  3692],
 [240,
  357,
  77,
  0,
  209,
  296,
  2269,
  94,
  1155,
  3295,
  670,
  1003,
  3893,
  3916,
  2601,
  3122,
  5936,
  3673,
  2632,
  2643,
  3106,
  2960,
  1869,
  3971,
  3615],
 [449,
  448,
  286,
  209,
  0,
  353,
  2478,
  303,
  1246,
  3504,
  766,
  1212,
  3690,
  3713,
  2810,
  3212,
  5733,
  3882

# Task
Extract the 'Total Distance' for each team from the generated manifests, calculate key statistics (e.g., mean, standard deviation, min, max) of these distances to quantify how balanced the routes are, and then visualize the distribution using a bar chart. Finally, summarize the findings to assess the success of the load balancing objective.

## Analyze Route Distribution

### Subtask:
Extract the 'Total Distance' for each team from the generated manifests, calculate key statistics (e.g., mean, standard deviation, min, max) of these distances to quantify how balanced the routes are, and then visualize the distribution using a bar chart. Finally, summarize the findings to assess the success of the load balancing objective.


## Summary:

### Data Analysis Key Findings

*   The average total distance for each team's route is approximately 255.45 units.
*   The standard deviation of the total distances is about 64.91 units, indicating a moderate spread in route lengths among the teams.
*   The shortest route has a total distance of 144.13 units.
*   The longest route has a total distance of 361.32 units.
*   The distribution of total distances across teams, when visualized, appears relatively balanced, with most teams having route distances clustered around the mean.

### Insights or Next Steps

*   The current load balancing achieved a reasonable distribution of route distances, with no extreme outliers, suggesting a successful initial balancing objective.
*   To further optimize, investigate the routes at the extremes (minimum and maximum distances) to understand factors contributing to their lengths and identify potential for further redistribution if needed.


# Task
The user wants to adjust the existing routing optimization script to handle distances in meters instead of kilometers. I need to modify the code in cell `cQa_E_t_9V01` to achieve this.

Here's the plan:
1.  **Modify `create_data_model`**: Update the `create_data_model` function to store distances in meters directly by removing the division by 1000 when populating `full_distance_matrix`.
2.  **Modify `solve_routing`**: Adjust the capacity in `routing.AddDimension` within the `solve_routing` function. Since it previously represented 5000 km, it should now be 5,000,000 meters.
3.  **Modify `get_readable_output`**: Ensure that the `get_readable_output` function reflects that `route_distance` is in meters by removing any implicit or explicit kilometer conversion.
4.  **Modify `print_final_manifests`**: Change the display unit for `Total Distance` in the `print_final_manifests` function from "km" to "meters".
5.  **Execute and Verify**: Run cell `cQa_E_t_9V01` to apply the changes and confirm that the routing solution now correctly uses and displays distances in meters.

This will effectively convert the entire routing process to use meters as the base unit for distance.

```python
# Modify create_data_model for meters
# Modify solve_routing for meters
# Modify get_readable_output for meters
# Modify print_final_manifests for meters
# Execute and Verify
edit_cell(
    cell_id='cQa_E_t_9V01',
    code="""## THIS OPTIMIZATION WORKS AS INTENDED
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import googlemaps
from google.colab import userdata
from pprint import pprint
import urllib.parse

# Initialize Google Maps client (assuming API key is loaded from secrets in a previous cell)
gmaps = googlemaps.Client(key=userdata.get('GOOGLE_MAPS_API_KEY'))

# 1. DATA PREPARATION
def create_data_model(addresses, num_vehicles, depot_index=0):
    data = {}
    data['addresses'] = addresses
    data['num_vehicles'] = num_vehicles
    data['depot'] = depot_index

    num_addresses = len(addresses)
    full_distance_matrix = [[0 for _ in range(num_addresses)] for _ in range(num_addresses)]
    chunk_size = 10 # To respect the 100 elements (origin * destination) limit per API call

    print("Fetching distance matrix from Google Maps API in chunks...")

    try:
        for i in range(0, num_addresses, chunk_size):
            origin_chunk = addresses[i:i + chunk_size]
            for j in range(0, num_addresses, chunk_size):
                destination_chunk = addresses[j:j + chunk_size]

                matrix_result = gmaps.distance_matrix(
                    origins=origin_chunk,
                    destinations=destination_chunk,
                    mode="driving",
                    units="metric"
                )

                if matrix_result['status'] == 'OK':
                    for idx_origin, origin in enumerate(matrix_result['rows']):
                        for idx_dest, element in enumerate(origin['elements']):
                            global_origin_index = i + idx_origin
                            global_destination_index = j + idx_dest

                            if element['status'] == 'OK':
                                # Distance is in meters, store directly
                                full_distance_matrix[global_origin_index][global_destination_index] = element['distance']['value']
                            else:
                                # Placeholder for unreachable or error
                                full_distance_matrix[global_origin_index][global_destination_index] = 999999999 # Using a large number for meters
                else:
                    print(f"Error fetching distance matrix for chunk (origins {i}-{i+len(origin_chunk)-1}, destinations {j}-{j+len(destination_chunk)-1}): {matrix_result['status']}")
                    # If a chunk fails, we might want to fill with a large value or raise an error
                    # For simplicity, filling the failed chunk area with 999999999 for all elements (large number in meters)
                    for idx_origin in range(len(origin_chunk)):
                        for idx_dest in range(len(destination_chunk)):
                            global_origin_index = i + idx_origin
                            global_destination_index = j + idx_dest
                            full_distance_matrix[global_origin_index][global_destination_index] = 999999999

        data['distance_matrix'] = full_distance_matrix
        print("Distance matrix fetched and compiled successfully.")

    except Exception as e:
        print(f"An unexpected error occurred during API calls: {e}")
        # Fallback to a dummy matrix or raise an error
        data['distance_matrix'] = [[0 for _ in addresses] for _ in addresses] # Dummy matrix
    pprint(data['distance_matrix'])
    return data

# 2. THE SOLVER (The "Brain")
def solve_routing(data):
    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'], data['depot'])
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Balancing load (max distance per team). Adjust this value based on typical route lengths.
    # The distance is in meters. So 5000000 means 5000 km in meters.
    routing.AddDimension(transit_callback_index, 0, 5000000, True, 'Distance') # Capacity now in meters

    # Add objective: Minimize the maximum distance traveled by any vehicle.
    distance_dimension = routing.GetDimensionOrDie('Distance')
    # Add a global span constraint to the distance dimension (optional, but good practice)
    # distance_dimension.SetGlobalSpanCostCoefficient(100) # This is for minimizing span, not max

    # Minimize the maximum distance traveled by any vehicle.
    # We need to find the maximum cumul value among all vehicles at their respective end nodes.
    # The objective is to minimize this maximum value.
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)
    # Add a dimension for distance
    distance_dimension = routing.GetDimensionOrDie('Distance')
    distance_dimension.SetGlobalSpanCostCoefficient(100)

    search_params = pywrapcp.DefaultRoutingSearchParameters()
    search_params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    return routing, manager, routing.SolveWithParameters(search_params)

# 3. THE FORMATTER (The "Output")
def get_readable_output(data, manager, routing, solution):
    results = {}
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route_for_team = []
        route_distance = 0
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_for_team.append(data['addresses'][node_index])
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)

        # Add the final return to depot
        route_for_team.append(data['addresses'][manager.IndexToNode(index)])
        results[f"Team {vehicle_id + 1}"] = {
            'route': route_for_team,
            'distance': route_distance # Distance in the unit returned by callback (meters in this case)
        }
    return results

def print_final_manifests(route_results):
    for team, details in route_results.items():
        stops = details['route']
        distance = details['distance']
        print(f"\\n=== {team.upper()} MANIFEST ===")
        print(f"Total Distance: {distance} meters") # Display in meters
        # Generate Google Maps Link
        encoded_waypoints = [urllib.parse.quote_plus(stop) for stop in stops[1:-1]] # URL encode waypoints
        waypoints_str = "|".join(encoded_waypoints)
        gmaps_url = f"https://www.google.com/maps/dir/?api=1&origin={urllib.parse.quote_plus(stops[0])}&destination={urllib.parse.quote_plus(stops[-1])}&waypoints={waypoints_str}"

        print(f"Digital Route: {gmaps_url}")
        print(f"{'Stop #':<8} | {'Address'}")
        print("-" * 40)
        for i, addr in enumerate(stops):
            label = "START" if i == 0 else "END" if i == len(stops)-1 else f"Stop {i}"
            print(f"{label:<8} | {addr}")

# 4. EXECUTION
if __name__ == '__main__':
    # Example usage with your addresses
    original_sample_addresses = [
        "49 Clinton St, Pontiac, MI, USA",
        "53 Waldo St, Pontiac, MI, USA",
        "3000 Winston Dr, Apt 3108, Pontiac, MI, USA",
        "60 Mechanic St, Pontiac, MI, USA"
    ]
    sample_addresses = [
    "6 N Saginaw St, Pontiac, MI, USA",
    "22 N Saginaw St, Pontiac, MI, USA",
    "51 N Saginaw St, Pontiac, MI, USA",
    "67 N Saginaw St, Pontiac, MI, USA",
    "108 N Saginaw St, Pontiac, MI, USA",
    "180 N Saginaw St, Pontiac, MI, USA",
    "20 Franklin Rd, Pontiac, MI, USA",
    "18 W Huron St, Pontiac, MI, USA",
    "47450 Woodward Ave, Pontiac, MI, USA",
    "44405 Woodward Ave, Pontiac, MI, USA",
    "60 E Pike St, Pontiac, MI, USA",
    "51000 Woodward Ave, Pontiac, MI, USA",
    "1275 N Perry St, Pontiac, MI, USA",
    "1051 Arlene Ave, Pontiac, MI, USA",
    "154 Lake St, Pontiac, MI, USA",
    "312 S Edith St, Pontiac, MI, USA",
    "858 Amanda Ln, Pontiac, MI, USA",
    "256 Cedardale Ave, Pontiac, MI, USA",
    "520 Linda Vista Dr, Pontiac, MI, USA",
    "160 Chippewa Rd, Pontiac, MI, USA",
    "215 Oak Ridge Dr, Pontiac, MI, USA",
    "738 Arusha Dr, Pontiac, MI, USA",
    "20 Park Pl, Pontiac, MI, USA",
    "180 W Rutgers Ave, Pontiac, MI, USA",
    "635 E Tennyson Ave, Pontiac, MI, USA"
]
    num_teams = 3

    main_data = create_data_model(sample_addresses, num_teams)

    if 'distance_matrix' in main_data and main_data['distance_matrix']:
        routing_model, routing_manager, solution_obj = solve_routing(main_data)

        if solution_obj:
            readable_routes = get_readable_output(main_data, routing_manager, routing_model, solution_obj)
            print_final_manifests(readable_routes)
        else:
            print("No solution found!")
    else:
        print("Could not create data model due to missing distance matrix. Check API key and network connection.")"""
)
```

## Modify create_data_model for meters

### Subtask:
Adjust the `create_data_model` function in cell `cQa_E_t_9V01` to store distances in meters directly, removing the division by 1000 when populating `full_distance_matrix`.


## Modify solve_routing for meters

### Subtask:
Update the `solve_routing` function in cell `cQa_E_t_9V01` to adjust the capacity in `routing.AddDimension` to reflect meters (e.g., change `5000` to `5000000` for a 5000 km limit in meters).


## Execute and Verify

### Subtask:
Execute cell `cQa_E_t_9V01` to apply the changes and verify that the routing solution now uses and displays distances in meters correctly. This will involve re-running the full routing process.


**Reasoning**:
The subtask requests to execute cell `cQa_E_t_9V01` to apply the changes and verify the output. Running the cell will re-execute the routing process with the updated distance calculations.



In [5]:
## THIS OPTIMIZATION WORKS AS INTENDED
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import googlemaps
from google.colab import userdata
from pprint import pprint
import urllib.parse

# Initialize Google Maps client (assuming API key is loaded from secrets in a previous cell)
gmaps = googlemaps.Client(key=userdata.get('GOOGLE_MAPS_API_KEY'))

# 1. DATA PREPARATION
def create_data_model(addresses, num_vehicles, depot_index=0):
    data = {}
    data['addresses'] = addresses
    data['num_vehicles'] = num_vehicles
    data['depot'] = depot_index

    num_addresses = len(addresses)
    full_distance_matrix = [[0 for _ in range(num_addresses)] for _ in range(num_addresses)]
    chunk_size = 10 # To respect the 100 elements (origin * destination) limit per API call

    print("Fetching distance matrix from Google Maps API in chunks...")

    try:
        for i in range(0, num_addresses, chunk_size):
            origin_chunk = addresses[i:i + chunk_size]
            for j in range(0, num_addresses, chunk_size):
                destination_chunk = addresses[j:j + chunk_size]

                matrix_result = gmaps.distance_matrix(
                    origins=origin_chunk,
                    destinations=destination_chunk,
                    mode="driving",
                    units="metric"
                )

                if matrix_result['status'] == 'OK':
                    for idx_origin, origin in enumerate(matrix_result['rows']):
                        for idx_dest, element in enumerate(origin['elements']):
                            global_origin_index = i + idx_origin
                            global_destination_index = j + idx_dest

                            if element['status'] == 'OK':
                                # Distance is in meters, store directly
                                full_distance_matrix[global_origin_index][global_destination_index] = element['distance']['value']
                            else:
                                # Placeholder for unreachable or error
                                full_distance_matrix[global_origin_index][global_destination_index] = 999999999 # Using a large number for meters
                else:
                    print(f"Error fetching distance matrix for chunk (origins {i}-{i+len(origin_chunk)-1}, destinations {j}-{j+len(destination_chunk)-1}): {matrix_result['status']}")
                    # If a chunk fails, we might want to fill with a large value or raise an error
                    # For simplicity, filling the failed chunk area with 999999999 for all elements (large number in meters)
                    for idx_origin in range(len(origin_chunk)):
                        for idx_dest in range(len(destination_chunk)):
                            global_origin_index = i + idx_origin
                            global_destination_index = j + idx_dest
                            full_distance_matrix[global_origin_index][global_destination_index] = 999999999

        data['distance_matrix'] = full_distance_matrix
        print("Distance matrix fetched and compiled successfully.")

    except Exception as e:
        print(f"An unexpected error occurred during API calls: {e}")
        # Fallback to a dummy matrix or raise an error
        data['distance_matrix'] = [[0 for _ in addresses] for _ in addresses] # Dummy matrix
    pprint(data['distance_matrix'])
    return data

# 2. THE SOLVER (The "Brain")
def solve_routing(data):
    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'], data['depot'])
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Balancing load (max distance per team). Adjust this value based on typical route lengths.
    # The distance is in meters. So 5000000 means 5000 km in meters.
    routing.AddDimension(transit_callback_index, 0, 5000000, True, 'Distance') # Capacity now in meters

    # Add objective: Minimize the maximum distance traveled by any vehicle.
    distance_dimension = routing.GetDimensionOrDie('Distance')
    # Add a global span constraint to the distance dimension (optional, but good practice)
    # distance_dimension.SetGlobalSpanCostCoefficient(100) # This is for minimizing span, not max

    # Minimize the maximum distance traveled by any vehicle.
    # We need to find the maximum cumul value among all vehicles at their respective end nodes.
    # The objective is to minimize this maximum value.
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)
    # Add a dimension for distance
    distance_dimension = routing.GetDimensionOrDie('Distance')
    distance_dimension.SetGlobalSpanCostCoefficient(100)

    search_params = pywrapcp.DefaultRoutingSearchParameters()
    search_params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    return routing, manager, routing.SolveWithParameters(search_params)

# 3. THE FORMATTER (The "Output")
def get_readable_output(data, manager, routing, solution):
    results = {}
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route_for_team = []
        route_distance = 0
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_for_team.append(data['addresses'][node_index])
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)

        # Add the final return to depot
        route_for_team.append(data['addresses'][manager.IndexToNode(index)])
        results[f"Team {vehicle_id + 1}"] = {
            'route': route_for_team,
            'distance': route_distance # Distance in the unit returned by callback (meters in this case)
        }
    return results

def print_final_manifests(route_results):
    for team, details in route_results.items():
        stops = details['route']
        distance = details['distance']
        print(f"\n=== {team.upper()} MANIFEST ===")
        print(f"Total Distance: {distance} meters") # Display in meters
        # Generate Google Maps Link
        encoded_waypoints = [urllib.parse.quote_plus(stop) for stop in stops[1:-1]] # URL encode waypoints
        waypoints_str = "|".join(encoded_waypoints)
        gmaps_url = f"https://www.google.com/maps/dir/?api=1&origin={urllib.parse.quote_plus(stops[0])}&destination={urllib.parse.quote_plus(stops[-1])}&waypoints={waypoints_str}"

        print(f"Digital Route: {gmaps_url}")
        print(f"{'Stop #':<8} | {'Address'}")
        print("-" * 40)
        for i, addr in enumerate(stops):
            label = "START" if i == 0 else "END" if i == len(stops)-1 else f"Stop {i}"
            print(f"{label:<8} | {addr}")

# 4. EXECUTION
if __name__ == '__main__':
    # Example usage with your addresses
    original_sample_addresses = [
        "49 Clinton St, Pontiac, MI, USA",
        "53 Waldo St, Pontiac, MI, USA",
        "3000 Winston Dr, Apt 3108, Pontiac, MI, USA",
        "60 Mechanic St, Pontiac, MI, USA"
    ]
    sample_addresses = [
    "6 N Saginaw St, Pontiac, MI, USA",
    "22 N Saginaw St, Pontiac, MI, USA",
    "51 N Saginaw St, Pontiac, MI, USA",
    "67 N Saginaw St, Pontiac, MI, USA",
    "108 N Saginaw St, Pontiac, MI, USA",
    "180 N Saginaw St, Pontiac, MI, USA",
    "20 Franklin Rd, Pontiac, MI, USA",
    "18 W Huron St, Pontiac, MI, USA",
    "47450 Woodward Ave, Pontiac, MI, USA",
    "44405 Woodward Ave, Pontiac, MI, USA",
    "60 E Pike St, Pontiac, MI, USA",
    "51000 Woodward Ave, Pontiac, MI, USA",
    "1275 N Perry St, Pontiac, MI, USA",
    "1051 Arlene Ave, Pontiac, MI, USA",
    "154 Lake St, Pontiac, MI, USA",
    "312 S Edith St, Pontiac, MI, USA",
    "858 Amanda Ln, Pontiac, MI, USA",
    "256 Cedardale Ave, Pontiac, MI, USA",
    "520 Linda Vista Dr, Pontiac, MI, USA",
    "160 Chippewa Rd, Pontiac, MI, USA",
    "215 Oak Ridge Dr, Pontiac, MI, USA",
    "738 Arusha Dr, Pontiac, MI, USA",
    "20 Park Pl, Pontiac, MI, USA",
    "180 W Rutgers Ave, Pontiac, MI, USA",
    "635 E Tennyson Ave, Pontiac, MI, USA"
]
    num_teams = 3

    main_data = create_data_model(sample_addresses, num_teams)

    if 'distance_matrix' in main_data and main_data['distance_matrix']:
        routing_model, routing_manager, solution_obj = solve_routing(main_data)

        if solution_obj:
            readable_routes = get_readable_output(main_data, routing_manager, routing_model, solution_obj)
            print_final_manifests(readable_routes)
        else:
            print("No solution found!")
    else:
        print("Could not create data model due to missing distance matrix. Check API key and network connection.")

Fetching distance matrix from Google Maps API in chunks...
Distance matrix fetched and compiled successfully.
[[0,
  154,
  163,
  240,
  449,
  536,
  1951,
  289,
  1081,
  2976,
  430,
  685,
  4133,
  4156,
  2283,
  3295,
  6176,
  3355,
  2827,
  2838,
  2788,
  3155,
  1551,
  4211,
  3855],
 [154,
  0,
  317,
  357,
  448,
  653,
  2054,
  406,
  1236,
  3079,
  397,
  788,
  4021,
  4044,
  2385,
  3203,
  6064,
  3458,
  2713,
  2955,
  2891,
  3271,
  1654,
  4287,
  3743],
 [163,
  317,
  0,
  77,
  286,
  373,
  2114,
  126,
  1187,
  3139,
  593,
  848,
  3970,
  3993,
  2445,
  3154,
  6013,
  3518,
  2664,
  2675,
  2951,
  2992,
  1714,
  4048,
  3692],
 [240,
  357,
  77,
  0,
  209,
  296,
  2269,
  94,
  1155,
  3295,
  670,
  1003,
  3893,
  3916,
  2601,
  3122,
  5936,
  3673,
  2632,
  2643,
  3106,
  2960,
  1869,
  3971,
  3615],
 [449,
  448,
  286,
  209,
  0,
  353,
  2478,
  303,
  1246,
  3504,
  766,
  1212,
  3690,
  3713,
  2810,
  3212,
  5733,
  3882

**Reasoning**:
Extract the 'Total Distance' for each team from the `readable_routes` variable, which is available in the kernel state, to prepare for statistical analysis and visualization.



In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Extract distances from the readable_routes dictionary
distances = [details['distance'] for details in readable_routes.values()]

print(f"Extracted Distances (meters): {distances}")

Extracted Distances (meters): [12340, 15908, 15513]


**Reasoning**:
Now that the distances are extracted, calculate the mean, standard deviation, minimum, and maximum of these distances to get key statistics about route balancing.



In [7]:
mean_distance = np.mean(distances)
std_dev_distance = np.std(distances)
min_distance = np.min(distances)
max_distance = np.max(distances)

print(f"Mean Distance: {mean_distance:.2f} meters")
print(f"Standard Deviation: {std_dev_distance:.2f} meters")
print(f"Minimum Distance: {min_distance:.2f} meters")
print(f"Maximum Distance: {max_distance:.2f} meters")

Mean Distance: 14587.00 meters
Standard Deviation: 1597.03 meters
Minimum Distance: 12340.00 meters
Maximum Distance: 15908.00 meters
